# Create or append Virtual Icechunk Store from SHYFEM forecast NetCDF files

* This notebook appends Taranto SHYFEM forecast data to Icechunk store using date-based set logic.
* If no repo exists, a new one will be created with references to all the existing NetCDF files. 

**Append Methodology:**
1. **`set_repo`**: extract all dates currently present in the Icechunk store's `time` coordinate
2. **`set_cloud`**: Scan the S3 bucket for all available NOS files and extract their dates.
3. **`new_dates`**: Calculate the difference (`set_cloud - set_repo`) to determine exactly which days need to be written for creation or appended.


In [1]:
import warnings
import os
import pandas as pd
import fsspec
import icechunk
import xarray as xr
from obstore.store import S3Store
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry 
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# Load credentials
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/rsignell4.env')

# Configuration
storage_endpoint = 'https://pangeo-eosc-minioapi.vm.fedcloud.eu'
storage_bucket = 'rsignell4-protocoast'
storage_name = 'taranto-icechunk-v1'  # icechunk version 1 format
bucket_url = f"s3://{storage_bucket}"

# Setup Filesystem
fs = fsspec.filesystem('s3', anon=False, endpoint_url=storage_endpoint, 
                       skip_instance_cache=True, use_listings_cache=False)

In [3]:
fs.ls(f'{storage_bucket}/icechunk/')

['rsignell4-protocoast/icechunk/antsiranana-icechunk',
 'rsignell4-protocoast/icechunk/era5-evap-icechunk',
 'rsignell4-protocoast/icechunk/noaa-cdr-icechunk',
 'rsignell4-protocoast/icechunk/taranto-icechunk',
 'rsignell4-protocoast/icechunk/taranto-icechunk-test',
 'rsignell4-protocoast/icechunk/taranto-icechunk-v1',
 'rsignell4-protocoast/icechunk/taranto-icechunk2',
 'rsignell4-protocoast/icechunk/taranto-icechunk3']

In [4]:
# there were some missing files -- let's see if they are there now?
#fs.ls('rsignell4-protocoast/full_dataset/shyfem/taranto/forecast/20260129/taranto_nos_20260129_nc4.nc')

In [5]:
%%time
# are they anywhere?
#fs.glob(f'{storage_bucket}/**/taranto_nos_20260129_nc4.nc')

CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 6.2 μs


## Icechunk concepts

Think of icechunk like a **library**:

| Object | Role |
|---|---|
| `storage` | The building — the physical S3 bucket + path where icechunk metadata lives |
| `RepositoryConfig` | The rules — defines how the repo behaves, including where to find virtual chunk data |
| `Repository` | The catalog — versioned, transactional index of the dataset (like a git repo for arrays) |
| `Session` | A checkout — a consistent, point-in-time view of a branch (e.g. `"main"`) |
| `session.store` | The open book — the zarr-compatible interface that xarray reads from |

**What "virtual" means:** The icechunk repo doesn't copy the raw NetCDF files — it stores *references* to their chunks. The `VirtualChunkContainer` in the config tells icechunk where to fetch those original chunks from when requested, making the repo lightweight while still providing versioning and transactional writes.

In [6]:
# Define Icechunk Storage & Config
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{bucket_url}/",
        store=icechunk.s3_store(region="not-used", anonymous=False, s3_compatible=True, 
                                force_path_style=True, endpoint_url=storage_endpoint),
    ),
)

credentials = icechunk.containers_credentials({f"{bucket_url}/": icechunk.s3_credentials(anonymous=False)})

# Setup VirtualiZarr Registry
store_obj = S3Store(storage_bucket, endpoint_url=storage_endpoint,
                    region='not-used', virtual_hosted_style_request=False)
registry = ObjectStoreRegistry({bucket_url: store_obj})
parser = HDFParser()

### Step 1: Create Date Sets (`set_repo` vs `set_cloud`)

In [7]:
# --- 1. Get Dates from Icechunk Repo (set_repo) ---
try:
    repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
    session = repo.readonly_session("main")
    ds = xr.open_zarr(session.store, consolidated=False, chunks={})
    
    if 'time' in ds.coords:
        # Extract dates as YYYYMMDD strings
        dates = pd.to_datetime(ds.time.values) + pd.Timedelta(days=1)
        set_repo = set(dates.strftime('%Y%m%d'))
    else:
        set_repo = set()
        
except Exception as e:
    print(f"Repo access failed or empty ({e}). Assuming set_repo is empty.")
    repo = None
    set_repo = set()

print(f"set_repo: {len(set_repo)} dates found.")

# --- 2. Get Dates from Cloud Bucket (set_cloud) ---
print("Scanning S3 for NOS files...")
nos_files = fs.glob(f'{bucket_url}/full_dataset/shyfem/taranto/forecast/*/*nos*.nc')

set_cloud = set()
date_to_files_map = {} # Helper to quickly find files for a date later

for f in nos_files:
    # Path structure: .../forecast/YYYYMMDD/taranto_nos_YYYYMMDD_nc4.nc
    try:
        date_str = f.split('/')[-2] # Parent folder is date
        set_cloud.add(date_str)
        
        # Store both NOS and assume OUS follows same pattern
        base_dir = os.path.dirname(f)
        nos_path = f's3://{f}'
        ous_path = f's3://{base_dir}/taranto_ous_{date_str}_nc4.nc'
        
        date_to_files_map[date_str] = {'nos': nos_path, 'ous': ous_path}
        
    except IndexError:
        pass

print(f"set_cloud: {len(set_cloud)} dates found.")

set_repo: 171 dates found.
Scanning S3 for NOS files...
set_cloud: 173 dates found.


In [8]:
# --- 3. Determine New Dates ---
new_dates = sorted(list(set_cloud - set_repo))

print(f"Dates to process: {len(new_dates)}")
if new_dates:
    print(f"Range: {new_dates[0]} to {new_dates[-1]}")

Dates to process: 2
Range: 20260319 to 20260320


### Step 2: Virtualize and Merge New Data

In [9]:
def fix_ds(ds):
    """Standardizes dimensions and coordinates for Taranto dataset."""
    ds = ds.rename_vars(time='valid_time')
    ds = ds.rename_dims(time='step')
    step = (ds.valid_time - ds.valid_time[0]).assign_attrs({"standard_name": "forecast_period"})
    time = ds.valid_time[0].assign_attrs({"standard_name": "forecast_reference_time"})
    ds = ds.assign_coords(step=step, time=time)
    ds = ds.drop_indexes("valid_time")
    ds = ds.drop_vars('valid_time')
    ds = ds.set_coords(['latitude', 'longitude', 'element_index', 'topology', 'total_depth'])
    return ds

In [10]:
ds_final = None

# new_dates = new_dates[:3]  # only if testing

if new_dates:
    # Reconstruct file lists based on the identified dates
    nos_urls = [date_to_files_map[d]['nos'] for d in new_dates]
    ous_urls = [date_to_files_map[d]['ous'] for d in new_dates]

    # --- Process NOS ---
    print(f"Virtualizing {len(nos_urls)} NOS files...")
    nos_list = [
        open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=["time"])
        for url in nos_urls
    ]
    nos_list = [fix_ds(ds) for ds in nos_list]
    combined_nos = xr.concat(
        nos_list, dim="time", coords="minimal", compat="override", combine_attrs="override"
    )

    # --- Process OUS ---
    print(f"Virtualizing {len(ous_urls)} OUS files...")
    ous_list = [
        open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=["time"])
        for url in ous_urls
    ]
    ous_list = [fix_ds(ds) for ds in ous_list]
    combined_ous = xr.concat(
        ous_list, dim="time", coords="minimal", compat="override", combine_attrs="override"
    )

    # --- Merge ---
    ds_final = xr.merge([combined_nos, combined_ous], compat='override')
    print("Datasets merged and ready for writing to Icechunk.")
    
else:
    print("No new dates found. Skipping processing.")

Virtualizing 2 NOS files...


AttributeError: 'builtins.PyObjectStoreConfig_S3Compatible' object has no attribute 'head'

### Step 3: Append to Icechunk

In [ ]:
if ds_final is not None:
    # Ensure we have a valid repo object
    if repo is None:
        repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
        initial_session = repo.writable_session("main")

        # Append
        print(f"Writing {len(ds_final.time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(initial_session.store)
    
        # Commit
        msg = f"Initialized with forecast data: {new_dates[0]} to {new_dates[-1]}"
        initial_session.commit(msg)
        print(f"Commit successful: '{msg}'")
    # Create Writable Session
    else:
        append_session = repo.writable_session("main")

        # Append
        print(f"Appending {len(ds_final.time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(append_session.store, append_dim="time")
    
        # Commit
        msg = f"Appended forecast data: {new_dates[0]} to {new_dates[-1]}"
        append_session.commit(msg)
        print(f"Commit successful: '{msg}'")

    # Verify History
    history = repo.ancestry(branch="main")
    latest = next(history)
    print(f"Latest Commit [{latest.written_at}]: {latest.message}")
    
else:
    print("Nothing to append.")